# Step 17 — audit: what left each site?

**Data type: RNA_array** (GSE65391). **Reads:** `data/run_artifacts/messages_*.csv`, written by
`send()` in steps 04–16. **Writes:** `step17_audit.csv`.

Every message a site sent passed through `send()`, which recorded the step, the site, what was sent,
and the shape of every piece. This notebook reads the whole record and checks the rule: **no message
carries one value per patient.** The check flags any piece with a dimension equal to the number of
patients at the sending site. A flag is then inspected by hand, because a vector can match that
number by coincidence.

In [1]:
source("../src/paths.R")
files <- sort(list.files(ARTIFACTS, pattern = "^messages_[0-9]+\\.csv$", full.names = TRUE))
log <- do.call(rbind, lapply(files, function(f)
  cbind(step = sub("messages_([0-9]+)\\.csv", "\\1", basename(f)), read.csv(f, stringsAsFactors = FALSE))))
nrow(log)

[1] 4998

## What each step sent

In [2]:
per_what <- aggregate(numbers ~ step + what + piece + dims, data = log, FUN = length)
names(per_what)[5] <- "messages"
per_what$numbers_each <- aggregate(numbers ~ step + what + piece + dims, data = log, FUN = max)$numbers
per_what[order(per_what$step), ]

,step,what,piece,dims,messages,numbers_each
,<chr>,<chr>,<chr>,<chr>,<int>,<int>
63,04,gene varies at site (flag),value,28948,3,28948
17,06,"per-gene n, sum, sum of squares",n,1,3,1
48,06,"per-gene n, sum, sum of squares",sum,27984,3,27984
49,06,"per-gene n, sum, sum of squares",sumsq,27984,3,27984
16,07,"per-gene n, mean, variance",n,1,3,1
27,07,mean |ComBat - location/scale|,value,1,3,1
39,07,"per-gene n, mean, variance",mean,27984,3,27984
62,07,"per-gene n, mean, variance",var,27984,3,27984
1,08,comparison sums 04,abs_err,1,3,1


In [3]:
aggregate(cbind(numbers, messages = 1) ~ step, data = log, FUN = sum)

step,numbers,messages
<chr>,<dbl>,<dbl>
04,86844,3
06,167907,9
07,167910,12
08,127162290,1599
09,9,3
10,903,12
11,5932953,303
12,1500,15
13,29004,3009


## The check

In [4]:
flagged <- log[log$patient_dimension, ]
if (nrow(flagged)) unique(flagged[, c("step", "site", "what", "piece", "dims")]) else cat("no piece has a patient dimension\n")

no piece has a patient dimension


In [5]:
write.csv(per_what, art("step17_audit.csv"), row.names = FALSE)

## Findings

Every message is one of these: a per-gene summary (counts, sums, sums of squares, flags); a genes ×
components block from the federated PCA; a per-cluster count or sum; a histogram; or a contingency
table. **None carries one value per patient**, and no site ever sent a gene × gene matrix.

By volume, most of what was sent is the PCA iteration. Each pass sends a genes × 20 block, and step 08
ran four PCAs on 8,825 genes to draw its picture. A few kilobytes of per-gene sums carry the batch
correction and the interferon score.